In [1]:
import tiktoken
from pathlib import Path
import PyPDF2
import os
from dotenv import load_dotenv
import logging
logger = logging.getLogger(__name__)   

In [2]:
try:
    os.chdir("../../agent/")
    os.getcwd()
    load_dotenv()
    from documents import DocumentProcessor, EmailHandler, DocxDocument, PDFHandler
    from models import WriteDocx, WriteEmail
    os.chdir("../data/TOSL-2024-103311/")
except Exception as e:
    print(f"Error changing directory or importing modules: {e}")

In [3]:
import re
from datetime import datetime
def parse_email_file(filepath: str) -> dict:
    """
    Parser en tekstfil med e-post på norsk format som i dine eksempler.
    Returnerer dict med: from_addr, to, cc, bcc, subject, date, body
    """
    with open(filepath, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    result = {
        'from_addr': None,
        'to': [],
        'cc': [],
        'bcc': [],
        'subject': None,
        'date': None,
        'body': ''
    }

    in_body = False
    body_lines = []

    for line in lines:
        line = line.rstrip('\n')

        if not line.strip():
            # tom linje → ofte start på body
            if not in_body and result['subject'] is not None:
                in_body = True
            continue

        if in_body:
            body_lines.append(line)
            continue

        # Header-parsing
        lower = line.lower()

        if lower.startswith('fra:'):
            # Fra: Anders Kristiansen <anders.kristiansen@email.no>
            match = re.search(r'<([^>]+)>', line)
            if match:
                result['from_addr'] = match.group(1).strip()
            else:
                # fallback – ta alt etter :
                result['from_addr'] = line.split(':', 1)[1].strip()

        elif lower.startswith('til:'):
            # Til: Carl Danielsen <carl.danielsen@email.no>
            # eller flere adresser adskilt med komma
            recipients = line.split(':', 1)[1].strip()
            result['to'] = [e.strip() for e in recipients.split(',') if e.strip()]

        elif lower.startswith(('kopi:', 'cc:', 'kopia:')):
            cc_str = line.split(':', 1)[1].strip()
            result['cc'] = [e.strip() for e in cc_str.split(',') if e.strip()]

        elif lower.startswith(('bcc:', 'blindkopi:')):
            bcc_str = line.split(':', 1)[1].strip()
            result['bcc'] = [e.strip() for e in bcc_str.split(',') if e.strip()]

        elif lower.startswith('emne:') or lower.startswith('subject:'):
            result['subject'] = line.split(':', 1)[1].strip()
        

        elif lower.startswith('dato:'):
            date_str = line.split(':', 1)[1].strip()
            # Forsøk å parse vanlige norske datoformater
            try:
                # 8. mai 2020, 15:42
                dt = datetime.strptime(date_str, '%d. %B %Y, %H:%M')
                result['date'] = dt
            except ValueError:
                try:
                    # fallback – ISO eller andre varianter
                    dt = datetime.fromisoformat(date_str.replace(' ', 'T'))
                    result['date'] = dt
                except:
                    result['date'] = date_str  # behold som streng hvis parsing feiler

    if body_lines:
        result['body'] = '\n'.join(body_lines).strip()
    
    if not result["subject"]:
        result["subject"] = "(Ingen emne)"
    if not result["body"]:
        result["body"] = "(Ingen innhold)"
    if not result["date"]:
        result["date"] = "(Ingen dato)"

    return result

def handle_eml_file(filepath: str):
    try:
        file = Path(filepath)
        if not file.is_file():
            logger.error(f"File not found: {filepath}")
            return
        eml_data = parse_email_file(file)
        email_data = WriteEmail.model_validate(eml_data)
        eml = EmailHandler().mk_eml(email_data)
        with open(f"./01_fabricated/{file.stem}.eml", "wb") as f:
            f.write(eml)

        target_dir = Path("./01_fabricated/old_txt")
        target_dir.mkdir(parents=True, exist_ok=True)
        new_path = target_dir / file.name
        file.rename(new_path)
        logger.info(f"Processed {file.name} → {file.stem}.eml, moved original to {new_path}")
    except Exception as e:
        logger.error(f"Error processing {filepath}: {e}")


In [9]:
import re
from pathlib import Path
from datetime import datetime
import os
files = []
start_date = datetime(2024, 2, 1)
end_date   = datetime(2024, 2, 20)

# Valgfritt: sjekk rekkefølge
if start_date > end_date:
    raise ValueError("Start date must be before or equal to end date")

pattern = re.compile(r'\d{4}-\d{2}-\d{2}')

for file in Path("./01_fabricated").glob("*.txt"):
    match = pattern.search(file.name)
    if match:
        try:
            file_date_str = match.group(0)
            file_date = datetime.fromisoformat(file_date_str)
            
            # ← Dette er den korrekte betingelsen når du har både start og slutt
            if start_date <= file_date < end_date:   # eller <= end_date hvis du vil inkludere 20. feb
                files.append(str(file))              # eller f"../01_fabricated/{file.name}"
                
        except ValueError:
            # ugyldig datoformat i filnavnet → hopp over
            continue

files

['01_fabricated/2024-02-15_19_svar_forsikring_reklamasjon_2_og_3_2024-02-15.txt',
 '01_fabricated/2024-02-19_10_pristilbud_nordby_varmekabler_2024-02-19.txt',
 '01_fabricated/2024-02-15_02_brev_oslo_brann_2024-02-15.txt']

In [1]:
from test_pipeline_manuscript_TOSL_2024_103311 import MANUSCRIPT
import json
with open("manuscript-TOSL-2024-103311.json", "w") as f:
    json.dump(MANUSCRIPT, f, indent=4)